# 🍽️ Augmentation Dataset — 10 images par plat
## 7 transformations PIL soignées + 3 ajouts IA d'ingrédients

**Principe :**
- **7 PIL** : mélanges optimaux de rotation légère + luminosité + couleur + recadrage
  → Pas de zoom excessif, image lisible et propre
- **3 SD** : ajout d'ingrédients spécifiques au plat (viande, sauce, légumes...)
  → SD détecte la zone la plus vide et y ajoute l'élément

**⚠️ AVANT DE COMMENCER :** Exécution → Modifier le type → **GPU T4**

**Structure ZIP attendue :**
```
dataset.zip
├── foutou_site.xlsx
└── foutou/
    ├── foutou_009.jpg
    └── ...
```

In [ ]:
!pip install diffusers transformers accelerate xformers torch Pillow pandas openpyxl -q
print('✅ Dépendances installées')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.3 MB/s eta 0:00:00
✅ Dépendances installées


In [ ]:
import torch
if torch.cuda.is_available():
    print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('❌ Pas de GPU — Exécution > Modifier le type > GPU T4')

✅ GPU : Tesla T4
   VRAM : 15.6 GB


In [ ]:
from google.colab import files
import zipfile, os

print('📁 Upload ton ZIP (xlsx + dossier images/)')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f'✅ Extrait : {fname}')
    elif fname.endswith('.xlsx'):
        print(f'✅ Excel : {fname}')
import subprocess; subprocess.run(['ls', '-la'])

📁 Upload ton ZIP (xlsx + dossier images/)


Saving Colabfile.zip to Colabfile.zip
✅ Extrait : Colabfile.zip


CompletedProcess(args=['ls', '-la'], returncode=0)

In [ ]:
# ═══════════════════════════════════════════════
# CONFIGURATION — adapter ces 4 lignes
# ═══════════════════════════════════════════════
EXCEL_INPUT   = 'Colabfile/image_site.xlsx'   # ← ton fichier Excel
SHEET_INDEX   = 0                    # 0=feuille1, 1=feuille2
IMAGE_COL     = 'image_id'           # colonne nom de l'image
IMAGES_FOLDER = 'Colabfile/images/'           # ← ton dossier images

OUTPUT_FOLDER = 'Colabfile/images_augmentees/'
EXCEL_OUTPUT  = 'Colabfile/annotations_augmentees.xlsx'
SD_MODEL      = 'stable-diffusion-v1-5/stable-diffusion-inpainting'
IMG_SIZE      = 512
NUM_STEPS     = 40
GUIDANCE      = 11.0
STRENGTH      = 0.88

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print('✅ Config OK — 7 PIL + 3 SD = 10 images par photo')

✅ Config OK — 7 PIL + 3 SD = 10 images par photo


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 7 TRANSFORMATIONS PIL — mélanges optimaux
#
# Règles appliquées :
#   • Zoom max 1.25x (recadrage visible mais plat encore lisible)
#   • Rotations légères (≤15°) ou 90°/180° (nets et propres)
#   • Chaque recette combine exactement 3 effets complémentaires
#   • On alterne : géométrie + éclairage + couleur
# ═══════════════════════════════════════════════════════════════════
from PIL import Image, ImageEnhance, ImageOps, ImageFilter, ImageDraw
import numpy as np

def bord(img):
    arr = np.array(img.convert('RGB'))
    b   = np.concatenate([arr[0], arr[-1], arr[:,0], arr[:,-1]])
    return tuple(b.mean(axis=0).astype(int))

def rot(img, angle):
    """Rotation douce — remplissage couleur dominante, pas de bandes noires."""
    return img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=bord(img))

def zoom_doux(img, factor=1.18):
    """Zoom léger centré — 1.18x max pour garder l'image lisible."""
    w, h = img.size
    cut  = int(min(w, h) * (1 - 1/factor) / 2)
    return img.crop((cut, cut, w-cut, h-cut)).resize((w, h), Image.LANCZOS)

def recadre(img, gauche=0, haut=0, droite=0, bas=0):
    """Recadrage asymétrique léger — change la composition sans perdre le plat."""
    w, h = img.size
    return img.crop((
        int(w*gauche), int(h*haut),
        int(w*(1-droite)), int(h*(1-bas))
    )).resize((w, h), Image.LANCZOS)

def couleur(img, r=0, g=0, b=0):
    ri, gi, bi = img.convert('RGB').split()
    ri = ri.point(lambda x: min(255, max(0, x+r)))
    gi = gi.point(lambda x: min(255, max(0, x+g)))
    bi = bi.point(lambda x: min(255, max(0, x+b)))
    return Image.merge('RGB', (ri, gi, bi))

def vignette_douce(img, force=70):
    """Vignette très douce — assombrissement subtil des bords."""
    w, h   = img.size
    cx, cy = w/2, h/2
    mask   = Image.new('L', (w, h))
    mask.putdata([
        max(0, int(255 - force * ((x-cx)**2/cx**2 + (y-cy)**2/cy**2)**0.5))
        for y in range(h) for x in range(w)
    ])
    return Image.composite(img.convert('RGB'), Image.new('RGB', (w,h), (0,0,0)), mask)

def spot_lateral(img, direction='gauche', force=0.38):
    """Éclairage venant d'un côté — effet naturel photo de restaurant."""
    w, h = img.size
    pixels = []
    for y in range(h):
        for x in range(w):
            if   direction == 'gauche': t = x/w
            elif direction == 'droite': t = 1-x/w
            elif direction == 'haut':   t = y/h
            else:                       t = 1-y/h
            pixels.append(min(255, max(0, int(255*(1-force*(1-t))))))
    mask = Image.new('L', (w,h))
    mask.putdata(pixels)
    return Image.composite(img.convert('RGB'), Image.new('RGB',(w,h),(0,0,0)), mask)

lum = lambda img, f: ImageEnhance.Brightness(img).enhance(f)
sat = lambda img, f: ImageEnhance.Color(img).enhance(f)
ctr = lambda img, f: ImageEnhance.Contrast(img).enhance(f)
net = lambda img, f: ImageEnhance.Sharpness(img).enhance(f)

# ── Les 7 recettes PIL ──────────────────────────────────────────────
PIL_RECETTES = [

    # 01 — Golden hour : légèrement chaud + lumineux + zoom doux
    #      Simule une photo prise en fin de journée avec belle lumière
    ('golden_hour',
     lambda img: sat(lum(couleur(zoom_doux(img, 1.15), r=40, g=10, b=-30), 1.25), 1.45)),

    # 02 — Ambiance sombre et contrastée + vignette
    #      Style photo de restaurant à éclairage tamisé
    ('sombre_contraste',
     lambda img: vignette_douce(ctr(lum(img, 0.65), 1.7), force=110)),

    # 03 — Rotation 90° + tons froids et nets
    #      Angle de vue différent, ambiance fraîche
    ('rot90_froid',
     lambda img: net(sat(couleur(rot(img, 90), r=-25, b=35), 0.85), 2.5)),

    # 04 — Miroir + éclairage latéral droit + chaud
    #      Composition inversée avec lumière dramatique
    ('miroir_spot_chaud',
     lambda img: couleur(spot_lateral(ImageOps.mirror(img), 'droite', 0.42), r=35, g=8, b=-28)),

    # 05 — Rotation -10° + recadrage léger + saturé vibrant
    #      Légère inclinaison, photo plus dynamique
    ('incline_vibrant',
     lambda img: sat(recadre(rot(img, -10), gauche=0.05, haut=0.05), 1.8)),

    # 06 — Rotation 180° + lumière du haut + désaturé élégant
    #      Vue "retournée" avec ambiance studio
    ('rot180_studio',
     lambda img: sat(spot_lateral(rot(img, 180), 'haut', 0.40), 0.55)),

    # 07 — Zoom doux + vignette + tons très chauds
    #      Gros plan appétissant style magazine culinaire
    ('magazine_chaud',
     lambda img: vignette_douce(couleur(zoom_doux(img, 1.22), r=50, g=15, b=-40), force=80)),
]

assert len(PIL_RECETTES) == 7, f'Attendu 7, trouvé {len(PIL_RECETTES)}'
print(f'✅ 7 recettes PIL définies :')
for i, (nom, _) in enumerate(PIL_RECETTES, 1):
    print(f'   {i}. {nom}')

✅ 7 recettes PIL définies :
   1. golden_hour
   2. sombre_contraste
   3. rot90_froid
   4. miroir_spot_chaud
   5. incline_vibrant
   6. rot180_studio
   7. magazine_chaud


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3 AJOUTS SD PAR PLAT — ingrédients spécifiques et cohérents
#
# Pour chaque plat, 3 ajouts qui ont du sens culinairement.
# Le prompt décrit précisément ce qu'on ajoute dans la zone libre.
# ═══════════════════════════════════════════════════════════════════

MOTS_CLES = {
    'attieke_poisson': ['attieke poisson', 'attiéké poisson', 'attieke_poisson'],
    'garba':           ['garba', 'attieke thon', 'attiéké thon', 'attieke_thon'],
    'foutou':          ['foutou', 'foutou banane', 'foutou igname', 'sauce graine'],
    'kedjenou':        ['kedjenou', 'kédjenou'],
    'gombo':           ['gombo', 'soupe gombo', 'sauce gombo'],
    'thieboudienne':   ['thieboudienne', 'thiéboudiène', 'thiebu', 'riz poisson'],
    'mafe':            ['mafe', 'mafé', 'ragout arachide'],
    'yassa':           ['yassa'],
    'thiakry':         ['thiakry', 'lait caille', 'lait caillé'],
    'domoda':          ['domoda'],
}

# ═══════════════════════════════════════════════════════════════════
# 3 AJOUTS SD PAR PLAT — ingrédients spécifiques et cohérents
#
# Pour chaque plat, 3 ajouts qui ont du sens culinairement.
# Le prompt décrit précisément ce qu'on ajoute dans la zone libre.
# ═══════════════════════════════════════════════════════════════════

MOTS_CLES = {
    'attieke_poisson': ['attieke poisson', 'attiéké poisson', 'attieke_poisson'],
    'garba':           ['garba', 'attieke thon', 'attiéké thon', 'attieke_thon'],
    'foutou':          ['foutou', 'foutou banane', 'foutou igname', 'sauce graine'],
    'kedjenou':        ['kedjenou', 'kédjenou'],
    'gombo':           ['gombo', 'soupe gombo', 'sauce gombo'],
    'thieboudienne':   ['thieboudienne', 'thiéboudiène', 'thiebu', 'riz poisson'],
    'mafe':            ['mafe', 'mafé', 'ragout arachide'],
    'yassa':           ['yassa'],
    'thiakry':         ['thiakry', 'lait caille', 'lait caillé'],
    'domoda':          ['domoda'],
}

SD_AJOUTS = {

    'foutou': [
        {'id': 'gros_poisson_fume',
         'label': 'Poisson fumé entier',
         'prompt': 'A whole large smoked dried fish (bonga) naturally placed on the side of the plate, dark brown smoky texture, blending perfectly with the ambient lighting, professional food photography.'},
        {'id': 'sauce_abond',
         'label': 'Plus de sauce graine',
         'prompt': 'A generous, abundant extra pour of thick glossy orange-red palm nut sauce (sauce graine) naturally spreading over the dish, rich textures, authentic West African food style, realistic food photo.'},
        {'id': 'gros_morceaux_boeuf',
         'label': 'Gros morceaux de bœuf',
         'prompt': 'Large succulent chunks of braised beef with glistening dark sauce, neatly arranged on the side, matching the plate style, natural lighting, realistic food photo.'},
    ],

    'garba': [
        {'id': 'gros_thon_frit',
         'label': 'Gros pavé de thon frit',
         'prompt': 'A massive, prominent piece of golden-brown fried tuna fish steak placed right next to the attieke, crispy texture, glistening with oil, perfectly integrated into the local Ivorian garba presentation, realistic photo.'},
        {'id': 'alloco_vibrant',
         'label': 'Alloco doré généreux',
         'prompt': 'A large side portion of golden fried ripe plantains (alloco) with sweet caramelized edges, piled naturally on a small plate next to the main dish, realistic reflections, professional culinary photography.'},
        {'id': 'avalanche_piment_oignon',
         'label': 'Avalanche oignons et piments',
         'prompt': 'An abundant heap of freshly chopped raw red and white onions mixed with sliced green and red hot habanero chili peppers, scattered generously around, vivid natural colors, realistic food photo.'},
    ],

    'attieke_poisson': [
        {'id': 'enorme_carpe_frite',
         'label': 'Énorme poisson frit',
         'prompt': 'A giant whole fried tilapia fish, crispy scored skin, beautifully golden, taking up significant space next to the attieke, blending naturally with the shadows, high-end African restaurant photography.'},
        {'id': 'alloco_cote',
         'label': 'Gros bol d alloco',
         'prompt': 'A generous mound of sweet fried plantain slices (alloco), dark golden color, shimmering naturally under the restaurant lights, placed perfectly in the background space.'},
        {'id': 'crevettes_geantes',
         'label': 'Grosses crevettes grillées',
         'prompt': 'A row of large, prominent grilled gambas prawns, pink with authentic char marks, arranged elegantly on the table surface with soft natural shadows.'},
    ],

    'kedjenou': [
        {'id': 'gros_morceaux_poulet',
         'label': 'Gros morceaux de poulet',
         'prompt': 'Large, chunky pieces of traditional slow-cooked braised chicken glistening with a rich, aromatic tomato and onion reduction, standing out sharply but naturally inside the clay pot style, realistic food photo.'},
        {'id': 'riz_blanc_volumineux',
         'label': 'Grand bol de riz blanc',
         'prompt': 'A large, steaming ceramic bowl filled with fluffy, perfectly separated white rice, standing prominently next to the stew, soft steam, natural lighting.'},
        {'id': 'piments_frais_massifs',
         'label': 'Gros piments et tomates',
         'prompt': 'A striking cluster of large whole red and yellow fresh chili peppers alongside thick slices of raw tomatoes, laid elegantly on the wooden table surface with soft depth of field.'},
    ],

    'gombo': [
        {'id': 'crabe_crevettes_geantes',
         'label': 'Crabe et crevettes géantes',
         'prompt': 'A large whole cooked red crab claw and massive pink shrimp emerging prominently from the thick green okra soup, glossy textures, looking rich and spectacular, realistic food photo.'},
        {'id': 'gros_foutou_extra',
         'label': 'Boules de foutou massives',
         'prompt': 'Two large, smooth, perfectly shaped yellow plantain foutou balls sitting elegantly on a traditional plate next to the soup, realistic smooth texture, flawless integration.'},
        {'id': 'viande_peau_boeuf',
         'label': 'Gros morceaux de kanda',
         'prompt': 'Large, thick chunks of tender cooked beef skin (kanda) and meat shimmering in the rich sauce, highly visible, authentic West African culinary style.'},
    ],

    'thieboudienne': [
        {'id': 'enorme_tranche_poisson',
         'label': 'Énorme morceau de poisson',
         'prompt': 'A massive, thick centerpiece log of stuffed white fish (gof) sitting proudly on top of the reddish broken rice, authentic herb stuffing visible, perfectly catching the scene lighting.'},
        {'id': 'legumes_geants_cuits',
         'label': 'Légumes géants',
         'prompt': 'A prominent display of large cooked Senegalese vegetables: a whole thick carrot, a massive chunk of cassava, and a big wedge of soft cabbage, glistening with broth, realistic food photo.'},
        {'id': 'crevettes_royales_thiébou',
         'label': 'Crevettes royales',
         'prompt': 'A spectacular arrangement of large jumbo prawns, beautifully cooked and pink, topping the rich rice dish naturally, professional food styling.'},
    ],

    'mafe': [
        {'id': 'gros_blocs_agneau',
         'label': 'Gros blocs d agneau',
         'prompt': 'Huge, tender chunks of braised lamb shoulder buried richly in a thick, deep-orange peanut butter sauce (mafé), glossy sauce texture reflecting light naturally, realistic food photo.'},
        {'id': 'riz_casse_abondant',
         'label': 'Gros dôme de riz',
         'prompt': 'A very large, beautifully formed dome of steaming white broken rice sitting right next to the peanut stew, highly visible, natural shadows.'},
        {'id': 'legumes_racines_massifs',
         'label': 'Gros morceaux de courge',
         'prompt': 'Large, chunky pieces of sweet potato and bright orange pumpkin cooked soft, shining in the peanut sauce, adding major volume to the composition.'},
    ],

    'yassa': [
        {'id': 'montagne_oignons',
         'label': 'Montagne d oignons',
         'prompt': 'A massive, generous heap of sweet caramelized onions cooked with mustard and lemon, glistening intensely, dominating the chicken garnish, highly detailed and realistic.'},
        {'id': 'gros_poulet_braise',
         'label': 'Gros poulet braisé entier',
         'prompt': 'A whole large half-chicken, beautifully charred and grilled, resting proudly under the mustard-onion sauce, authentic texture, restaurant quality photography.'},
        {'id': 'olives_citrons_massifs',
         'label': 'Grosses tranches de citron',
         'prompt': 'Large juicy lemon wedges and a handful of plump green olives scattered artfully and generously over the dish, vibrant colors, realistic lighting.'},
    ],

    'thiakry': [
        {'id': 'avalanche_glacons',
         'label': 'Avalanche de glaçons',
         'prompt': 'A beautiful pile of large solid translucent ice cubes melting naturally next to the dessert bowl, clear water droplets, shiny ice reflections, perfectly integrated into the scene background, realistic photo.'},
        {'id': 'mangue_fraiche',
         'label': 'Mangue fraîche',
         'prompt': 'Generous thick slices of ripe bright orange mango arranged elegantly on a wooden surface next to the food, natural shadow, vivid colors, realistic food photo.'},
        {'id': 'coco_raisin',
         'label': 'Topping coco raisins',
         'prompt': 'A large rich heap of golden raisins and grated coconut flakes scattered naturally on the table, realistic shadows, fine details, high-end culinary photography.'},
    ],

    'domoda': [
        {'id': 'gros_morceaux_viande',
         'label': 'Gros morceaux de viande',
         'prompt': 'Large, substantial chunks of beef stew meat dripping with thick, smooth Gambian tomato-peanut sauce, highly noticeable, authentic texture, realistic food photo.'},
        {'id': 'legumes_courge_geante',
         'label': 'Gourd et patates massives',
         'prompt': 'Big, prominent chunks of cooked butternut squash and sweet potatoes arranged visibly in the rich sauce, vibrant orange tones, natural lighting.'},
        {'id': 'grand_plat_riz',
         'label': 'Grand accompagnement riz',
         'prompt': 'A big, rich portion of fluffy white rice filling out the empty background area cleanly, casting realistic soft shadows on the table.'},
    ],

    'generique': [
        {'id': 'viande_grillee_généreuse',
         'label': 'Gros morceaux de viande',
         'prompt': 'A generous pile of large, seasoned grilled meat chunks with beautiful caramelized edges, arranged visibly on a small rustic plate, realistic food photo.'},
        {'id': 'piments_citrons_massifs',
         'label': 'Gros piments et citrons',
         'prompt': 'A highly visible arrangement of fresh large red chili peppers and whole cut lemons sitting naturally on the table surface.'},
        {'id': 'gros_bol_sauce',
         'label': 'Gros bol de sauce riche',
         'prompt': 'A large, deep ceramic bowl filled with a rich, glossy West African dark sauce, placed prominently in the background with realistic focus.'},
    ],
}

def detecter_plat(row, image_id):
    # On regarde STRICTEMENT et UNIQUEMENT le nom de l'image (converti en minuscules)
    texte = str(image_id).lower()

    for cle, mots in MOTS_CLES.items():
        for mot in mots:
            if mot in texte:
                return cle
    return 'generique'

print(f'✅ {len(SD_AJOUTS)} plats configurés — 3 ajouts SD chacun')

def detecter_plat(row, image_id):
    # On regarde STRICTEMENT et UNIQUEMENT le nom de l'image (converti en minuscules)
    texte = str(image_id).lower()

    for cle, mots in MOTS_CLES.items():
        for mot in mots:
            if mot in texte:
                return cle
    return 'generique'

print(f'✅ {len(SD_AJOUTS)} plats configurés — 3 ajouts SD chacun')

✅ 11 plats configurés — 3 ajouts SD chacun


In [ ]:
from diffusers import StableDiffusionInpaintPipeline

print(f'Chargement du modèle...')
print('Premier lancement : téléchargement ~1.7GB (2-3 min)...')

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    SD_MODEL,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to('cuda')

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('✅ xFormers activé (économie VRAM)')
except:
    pass
pipe.enable_model_cpu_offload()
print('✅ Modèle prêt !')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Chargement du modèle...
Premier lancement : téléchargement ~1.7GB (2-3 min)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✅ xFormers activé (économie VRAM)
✅ Modèle prêt !


In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def find_image(folder, image_id):
    """Cherche l'image dans le dossier et ses sous-dossiers."""
    folder = Path(folder)
    # Recherche directe avec/sans extension
    for candidate in [folder / image_id] + \
                     [folder / f"{image_id}{e}" for e in [".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG"]]:
        if candidate.exists():
            return candidate
    # Sous-dossiers
    stem = Path(image_id).stem.lower()
    for f in folder.rglob("*"):
        if f.is_file() and f.stem.lower() == stem \
                and f.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            return f
    return None

def trouver_zone_libre(img):
    """
    Divise l'image en grille 3×3.
    Retourne le masque de la zone avec la variance la plus basse
    (= zone la plus uniforme = fond/espace vide autour du plat).
    Le plat (zone la plus chargée/variée) reste dans le noir.
    """
    arr  = np.array(img.convert('RGB'), dtype=float)
    h, w = arr.shape[:2]
    rows = np.array_split(np.arange(h), 3)
    cols = np.array_split(np.arange(w), 3)

    zones = []
    for ri, row_idx in enumerate(rows):
        for ci, col_idx in enumerate(cols):
            patch = arr[row_idx[0]:row_idx[-1], col_idx[0]:col_idx[-1]]
            zones.append({
                'variance': patch.var(),
                'x1': col_idx[0], 'y1': row_idx[0],
                'x2': col_idx[-1], 'y2': row_idx[-1],
                'nom': f'R{ri}C{ci}',
            })

    zones.sort(key=lambda z: z['variance'])
    return zones  # classées du plus libre au plus chargé

def make_masques_sd(img):
    """
    Crée 3 masques pour les 3 ajouts SD.
    On prend les 3 zones les plus libres (variance la plus basse).
    Blanc = zone où SD ajoute l'ingrédient.
    Noir  = reste de l'image (plat intact).
    """
    zones   = trouver_zone_libre(img)[:3]
    w, h    = img.size
    masques = []

    for z in zones:
        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)
        draw.rectangle([z['x1'], z['y1'], z['x2'], z['y2']], fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(14))
        masques.append((mask, z['nom']))

    return masques

NEG_PROMPT = (
    'blurry, deformed, ugly, cartoon, text, watermark, '
    'duplicate main dish, wrong food, distorted plate'
)

def sd_ajout(base_img, masque, prompt):
    return pipe(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        image=base_img,
        mask_image=masque,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE,
        strength=STRENGTH,
        width=IMG_SIZE, height=IMG_SIZE,
    ).images[0]

def exporter_excel(df_orig, df_new, chemin):
    combined = pd.concat([df_orig, df_new], ignore_index=True)
    with pd.ExcelWriter(chemin, engine='openpyxl') as writer:
        df_new.to_excel(writer,   sheet_name='Images_Augmentees',  index=False)
        combined.to_excel(writer, sheet_name='Toutes_Annotations', index=False)
        if 'source' in df_new.columns:
            r = (df_new.groupby(['plat_detecte', 'source', 'modification'])
                 .size().reset_index(name='count'))
            r.to_excel(writer, sheet_name='Resume', index=False)
        COLORS = {
            'Images_Augmentees': '1B4332',
            'Toutes_Annotations': '2D6A4F',
            'Resume': '40916C',
        }
        for sn, ws in writer.sheets.items():
            c = COLORS.get(sn, '2D6A4F')
            for cell in ws[1]:
                cell.fill = PatternFill('solid', fgColor=c)
                cell.font = Font(bold=True, color='FFFFFF', size=11)
                cell.alignment = Alignment(horizontal='center')
            for i, col in enumerate(ws.columns, 1):
                ml = max((len(str(x.value)) if x.value else 0) for x in col)
                ws.column_dimensions[get_column_letter(i)].width = min(ml + 4, 60)
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions

print('✅ Utilitaires prêts')

✅ Utilitaires prêts


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GÉNÉRATION PRINCIPALE
#
# Pour chaque image originale :
#   → 7 PIL  : transformations visuelles (copie, original intact)
#   → 3 SD   : ajouts d'ingrédients dans les zones libres détectées
# ═══════════════════════════════════════════════════════════════════
import glob as _glob

df = pd.read_excel(EXCEL_INPUT, sheet_name=SHEET_INDEX)
print(f'📊 {len(df)} images originales → {len(df)*10} à générer')

if IMAGE_COL not in df.columns:
    print(f'❌ Colonne "{IMAGE_COL}" introuvable')
    print(f'   Colonnes disponibles : {list(df.columns)}')
else:
    print('\n🔍 Détection des plats (aperçu) :')
    for _, row in df.head(4).iterrows():
        img_id = str(row[IMAGE_COL])
        print(f'   {img_id:35s} → {detecter_plat(row.to_dict(), img_id)}')
    print()

    new_rows = []
    errors   = []
    tot_pil  = 0
    tot_sd   = 0

    for idx, row in df.iterrows():
        image_id = str(row[IMAGE_COL]).strip()
        img_path = find_image(IMAGES_FOLDER, image_id)

        if img_path is None:
            print(f'⚠️  Image introuvable : {image_id}')
            errors.append(image_id)
            continue

        cle    = detecter_plat(row.to_dict(), image_id)
        ajouts = SD_AJOUTS.get(cle, SD_AJOUTS['generique'])
        stem   = Path(image_id).stem

        print(f'\n{"─"*62}')
        print(f'  [{idx+1}/{len(df)}]  {image_id}  →  {cle}')
        print(f'{"─"*62}')

        # Charger l'original UNE FOIS — ne sera jamais modifié
        base = Image.open(img_path).convert('RGB').resize(
            (IMG_SIZE, IMG_SIZE), Image.LANCZOS
        )

        # Détecter les 3 zones libres pour les ajouts SD
        masques_sd = make_masques_sd(base)
        zones_nom  = [z for _, z in masques_sd]
        print(f'  Zones libres détectées : {zones_nom}')

        # ── 7 transformations PIL ───────────────────────────────────
        print('  📐 PIL...')
        for i, (nom, fn) in enumerate(PIL_RECETTES, 1):
            try:
                out    = fn(base).convert('RGB')
                new_id = f'{stem}_pil{i:02d}_{nom}'
                out.save(Path(OUTPUT_FOLDER) / f'{new_id}.jpg', quality=96)

                nr = row.to_dict()
                nr[IMAGE_COL]              = f'{new_id}.jpg'
                nr['image_source']         = image_id
                nr['plat_detecte']         = cle
                nr['source']               = 'PIL'
                nr['modification']         = nom
                nr['confiance_annotation'] = 92
                new_rows.append(nr)
                tot_pil += 1
                print(f'    {i}/7 ✅  {nom}')
            except Exception as e:
                print(f'    {i}/7 ❌  {nom} — {str(e)[:50]}')
                errors.append(f'{image_id}_pil_{nom}')

        # ── 3 ajouts SD ────────────────────────────────────────────
        print('  🤖 SD ajouts...')
        for i, (ajout, (masque, zone_nom)) in enumerate(
            zip(ajouts[:3], masques_sd), 1
        ):
            print(f'    {i}/3  {ajout["label"]}  (zone {zone_nom}) ... ',
                  end='', flush=True)
            try:
                out    = sd_ajout(base, masque, ajout['prompt'])
                new_id = f'{stem}_sd{i:02d}_{ajout["id"]}'
                out.save(Path(OUTPUT_FOLDER) / f'{new_id}.jpg', quality=95)

                nr = row.to_dict()
                nr[IMAGE_COL]              = f'{new_id}.jpg'
                nr['image_source']         = image_id
                nr['plat_detecte']         = cle
                nr['source']               = 'SD_Inpainting'
                nr['modification']         = ajout['label']
                nr['zone_ajout']           = zone_nom
                nr['confiance_annotation'] = 82
                new_rows.append(nr)
                tot_sd += 1
                print('✅')
            except Exception as e:
                print(f'❌  {str(e)[:60]}')
                errors.append(f'{image_id}_sd_{ajout["id"]}')

        # Checkpoint toutes les 50 images originales traitées
        if (idx + 1) % 50 == 0 and new_rows:
            tmp = pd.DataFrame(new_rows)
            exporter_excel(
                df, tmp,
                EXCEL_OUTPUT.replace('.xlsx', '_checkpoint.xlsx')
            )
            print(f'\n  💾 Checkpoint : {tot_pil+tot_sd} images sauvegardées\n')

    print(f'\n{"═"*62}')
    print(f'  ✅  PIL  : {tot_pil}')
    print(f'  ✅  SD   : {tot_sd}')
    print(f'  ✅  TOTAL : {tot_pil+tot_sd}  →  {OUTPUT_FOLDER}')
    if errors:
        print(f'  ⚠️   Erreurs : {len(errors)}')
    print(f'{"═"*62}')

📊 5 images originales → 50 à générer

🔍 Détection des plats (aperçu) :
   foutou1.jpg                         → foutou
   foutou2.jpg                         → foutou
   attieke.jpg                         → generique
   thiakry.jpg                         → thiakry


──────────────────────────────────────────────────────────────
  [1/5]  foutou1.jpg  →  foutou
──────────────────────────────────────────────────────────────
  Zones libres détectées : ['R0C2', 'R2C0', 'R2C1']
  📐 PIL...
    1/7 ✅  golden_hour
    2/7 ✅  sombre_contraste
    3/7 ✅  rot90_froid
    4/7 ✅  miroir_spot_chaud
    5/7 ✅  incline_vibrant
    6/7 ✅  rot180_studio
    7/7 ✅  magazine_chaud
  🤖 SD ajouts...
    1/3  Poisson fumé entier  (zone R0C2) ... 

  0%|          | 0/35 [00:00<?, ?it/s]

✅
    2/3  Plus de sauce graine  (zone R2C0) ... 

  0%|          | 0/35 [00:00<?, ?it/s]

✅
    3/3  Gros morceaux de bœuf  (zone R2C1) ... 

...

✅
    2/3  Légumes cuits  (zone R1C1) ... 

  0%|          | 0/35 [00:00<?, ?it/s]

✅
    3/3  Crevettes  (zone R0C2) ... 

  0%|          | 0/35 [00:00<?, ?it/s]

✅

══════════════════════════════════════════════════════════════
  ✅  PIL  : 35
  ✅  SD   : 15
  ✅  TOTAL : 50  →  Colabfile/images_augmentees/
══════════════════════════════════════════════════════════════


In [ ]:
if new_rows:
    df_new = pd.DataFrame(new_rows)
    exporter_excel(df, df_new, EXCEL_OUTPUT)
    print(f'✅ Excel exporté : {EXCEL_OUTPUT}')
    print(f'   • Images_Augmentees  : {len(df_new)} nouvelles images')
    print(f'   • Toutes_Annotations : {len(df)+len(df_new)} au total')
    print(f'   • Resume             : répartition par plat et type')
else:
    print('⚠️  Aucune image générée — vérifie les erreurs ci-dessus')

✅ Excel exporté : Colabfile/annotations_augmentees.xlsx
   • Images_Augmentees  : 50 nouvelles images
   • Toutes_Annotations : 55 au total
   • Resume             : répartition par plat et type


In [ ]:
from google.colab import files
import glob as _glob

with zipfile.ZipFile('resultats.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for img in _glob.glob(f'{OUTPUT_FOLDER}*.jpg'):
        zf.write(img)
    if os.path.exists(EXCEL_OUTPUT):
        zf.write(EXCEL_OUTPUT)
    if os.path.exists('apercu.png'):
        zf.write('apercu.png')

print('✅ ZIP prêt — téléchargement automatique...')
files.download('resultats.zip')
files.download(EXCEL_OUTPUT)

✅ ZIP prêt — téléchargement automatique...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>